# SmartCampus AI — Phase C: Unsloth 4-bit QLoRA Fine-Tuning & `Q4_K_M` GGUF Export for Ollama

**Project:** SmartCampus AI (IIT / ISB + ISA extension, Groupe 3 *Night Agents*, NAU 2026–2027)
**Recommended Colab Pro Runtime:** GPU `L4 (24 GB)` or `A100 (40 GB)` (Account #4)

**End-to-End Pipeline:**
1. Load winning open-weight model (`unsloth/Qwen3-8B-unsloth-bnb-4bit` or `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`) in 4-bit NF4.
2. Fine-tune with **QLoRA** (`r=16`, `lora_alpha=32`) on our RAG-aware FR / Fusha / Tunisian Derja / Arabizi / HITL dataset (`Visitor -> Candidate`, `NAU Responsable`).
3. Save lightweight LoRA adapters (`~160 MB`) to Google Drive.
4. Merge + quantize and export **`Q4_K_M` GGUF (`~4.7–5.2 GB`)** + **`Modelfile`** (with exact `/no_think` `SYSTEM` directive and ChatML `TEMPLATE`) to Google Drive.
5. Smoke-test inside Ollama directly in Colab, then download `.gguf` to your PC and run `ollama create smartcampus-vision-8b -f Modelfile.smartcampus-vision-8b`.


In [ ]:
# Cell 1 — Mount Google Drive (when on Colab) & install Unsloth (Colab Pro GPU)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Running outside Google Colab; using local workspace paths.")

!pip install -q --no-deps bitsandbytes accelerate peft trl triton cut_cross_entropy unsloth_zoo
!pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install -q --no-deps unsloth


In [ ]:
# Cell 2 — Load Qwen in 4-bit NF4 and attach QLoRA adapters (r=16, alpha=32)
from pathlib import Path
import torch
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"  # Alternative: "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096
DRIVE_ROOT = (
    Path("/content/drive/MyDrive/SmartCampus_AI")
    if Path("/content/drive/MyDrive").exists()
    else Path("smartcampus_qlora_artifacts")
)
(DRIVE_ROOT / "data").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "adapters").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "gguf").mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Loaded:", BASE_MODEL, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Cell 3 — Load RAG-aware Tunisian SFT dataset (or generate aligned starter JSONL from official iit.tn sources)
import json
from datasets import load_dataset

DATASET_PATH = DRIVE_ROOT / "data" / "smartcampus_sft.jsonl"

STARTER_SYSTEM_HEADER = (
    "You are VISION, the SmartCampus AI academic orientation and pre-registration supervisor "
    "for IIT, ISB, and the ISA extension (NAU Sfax). Welcome every Visitor (high-school student, "
    "university student, or parent/relative) and authenticated Candidate in their exact register "
    "(French, Fusha, Tunisian Derja, or Arabizi). Always cite official sources using [Source: ...], "
    "never invent tuition fees, thresholds, or deadlines, enforce the 'Give or Keep' rule "
    "(Visitor -> Candidate), and never make admission decisions (only the human NAU Responsable decides)."
)

if not DATASET_PATH.exists():
    starter_rows = [
        {
            "conversations": [
                {
                    "role": "system",
                    "content": (
                        f"{STARTER_SYSTEM_HEADER}\n\n[OFFICIAL RAG EXCERPTS]\n"
                        "[Source: https://iit.tn/filieres-ingenieur/genie-informatique-2/, §Organisation] "
                        "Le Groupe Universitaire NAU (IIT Sfax) propose un diplôme d'ingénieur en Génie Informatique "
                        "accrédité ASIIN / Euro-Inf (niveau Master) avec trois parcours de spécialisation : "
                        "GLID (Génie Logiciel et Informatique Décisionnelle), ARSI (Administration Réseau et Sécurité "
                        "Informatique) et SDIA (Science des Données et Intelligence Artificielle). "
                        "Pré-requis : Prépa, Licence ou Mastère. Formation dispensée en français et en anglais."
                    ),
                },
                {
                    "role": "user",
                    "content": "3aslema, chnowa les parcours elli mawjoudin fel Cycle Ingénieur Génie Informatique fi l'IIT?",
                },
                {
                    "role": "assistant",
                    "content": (
                        "3aslema! Hasb el catalogue el rasmi [Source: https://iit.tn/filieres-ingenieur/genie-informatique-2/], "
                        "el Cycle Ingénieur Génie Informatique fi l'IIT Sfax (accrédité ASIIN / Euro-Inf) fih 3 spécialités : "
                        "GLID (Génie Logiciel w Informatique Décisionnelle), ARSI (Administration Réseau w Sécurité) "
                        "w SDIA (Science des Données w Intelligence Artificielle). El formation ta9ra bel français w anglais w maftou7a "
                        "lel Prépa, Licence walla Mastère. Tnajem t9olli chnowa diplômek tawa bech nwajhek?"
                    ),
                },
            ]
        },
        {
            "conversations": [
                {
                    "role": "system",
                    "content": (
                        f"{STARTER_SYSTEM_HEADER}\n\n[OFFICIAL RAG EXCERPTS]\n"
                        "[Source: https://iit.tn/test/procedure-et-frais-dinscription/, §Procédure d’admission] "
                        "La candidature aux filières d'ingénierie se fait par préinscription en ligne, suivie d'un "
                        "concours sur dossier, d'un entretien de validation et d'un test d'admission devant une commission "
                        "d'au moins trois enseignants permanents. À l'issue de ces entretiens, la commission / le NAU Responsable "
                        "déclare les dossiers acceptés, refusés ou mis en liste d'attente. Les modalités et tarifs officiels "
                        "sont confirmés directement par le secrétariat administratif IIT / NAU."
                    ),
                },
                {
                    "role": "user",
                    "content": "عسلامة، كان عندي معدل باهي نتقبل أوتوماتيك؟ وقداش بالضبط معاليم الترسيم في كل الاختصاصات؟",
                },
                {
                    "role": "assistant",
                    "content": (
                        "عسلامة! لا، ما فماش قبول أوتوماتيكي. حسب [Source: https://iit.tn/test/procedure-et-frais-dinscription/]، "
                        "القبول في مسالك الهندسة في الـ IIT يصير بدراسة ملف، مقابلة تقييم واختبار قبول قدام لجنة بشرية، "
                        "والمسؤول البشري في NAU واللجنة هوما وحدهم اللي ياخذو القرار النهائي (قبول، قائمة انتظار ولا رفض). "
                        "وبالنسبة للمعاليم والتعريفات الرسمية المحينة، لازم تتأكد منهم مباشرة عند الإدارة والـ NAU Responsable."
                    ),
                },
            ]
        },
        {
            "conversations": [
                {
                    "role": "system",
                    "content": (
                        f"{STARTER_SYSTEM_HEADER}\n\n[OFFICIAL RAG EXCERPTS]\n"
                        "[Source: https://iit.tn/test/procedure-et-frais-dinscription/, §Procédure d’admission] "
                        "La candidature aux filières d'ingénierie passe par une préinscription en ligne puis un entretien "
                        "de validation avec la commission humaine."
                    ),
                },
                {
                    "role": "user",
                    "content": "Bonjour, je suis parent d'un bachelier. Dois-je créer un compte dès maintenant pour poser des questions ?",
                },
                {
                    "role": "assistant",
                    "content": (
                        "Bonjour ! Non, conformément à la règle « Give or Keep », l'exploration des programmes et "
                        "l'orientation sont entièrement ouvertes sans compte pendant une session anonyme de 7 jours. "
                        "La connexion (par code OTP ou Google OAuth) n'est demandée qu'au moment de sauvegarder une shortlist, "
                        "de téléverser un relevé de notes officiel, de réserver un créneau d'entretien ou de lancer la "
                        "préinscription [Source: https://iit.tn/test/procedure-et-frais-dinscription/]. La décision finale "
                        "revient toujours au NAU Responsable humain."
                    ),
                },
            ]
        },
    ]
    with DATASET_PATH.open("w", encoding="utf-8") as f:
        for r in starter_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("Created aligned starter JSONL (replace with your full team SFT dataset!):", DATASET_PATH)

dataset = load_dataset("json", data_files=str(DATASET_PATH), split="train")

def format_prompts(examples):
    texts = []
    for conv in examples["conversations"]:
        try:
            txt = tokenizer.apply_chat_template(
                conv, tokenize=False, add_generation_prompt=False, enable_thinking=False
            )
        except TypeError:
            txt = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        texts.append(txt)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("Training examples:", len(dataset))


In [ ]:
# Cell 4 — Run 4-bit QLoRA Fine-Tuning (TRL SFTTrainer)
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=str(DRIVE_ROOT / "checkpoints" / "qwen_qlora"),
    ),
)

trainer_stats = trainer.train()
print("Training complete:", trainer_stats.metrics)

# Save lightweight LoRA adapter (~160 MB) first
ADAPTER_DIR = DRIVE_ROOT / "adapters" / "smartcampus-vision-8b-lora"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved LoRA adapter to:", ADAPTER_DIR)


In [ ]:
# Cell 5 — Merge LoRA + Export Q4_K_M GGUF (~4.7–5.2 GB) & Ollama Modelfile to Google Drive
GGUF_DIR = DRIVE_ROOT / "gguf" / "smartcampus-vision-8b"

model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

if getattr(tokenizer, "_ollama_modelfile", None):
    (GGUF_DIR / "Modelfile").write_text(tokenizer._ollama_modelfile, encoding="utf-8")

!ls -lh "{GGUF_DIR}"


In [ ]:
# Cell 6 (Optional) — Smoke-test your exported GGUF inside Ollama right here in Colab
# Aligned verbatim with ml/colab/Modelfile.smartcampus-vision-8b (exact SYSTEM /no_think + ChatML TEMPLATE)
import subprocess
import time

!curl -fsSL https://ollama.com/install.sh | sh
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

gguf_files = list(GGUF_DIR.glob("*.gguf"))
assert gguf_files, "No .gguf file found in GGUF_DIR!"
gguf_file = gguf_files[0]

MODELFILE_BODY = '''SYSTEM """You are VISION, the SmartCampus AI academic orientation supervisor for IIT, ISB, and ISA (NAU Sfax). Welcome every Visitor and Candidate in their exact register (French, Fusha, Tunisian Derja, or Arabizi), cite official catalog/RAG sources only, and never make admission decisions (only the human NAU Responsable decides).
/no_think"""

TEMPLATE """{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{- range .Messages }}{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ .Content }}<|im_end|>
{{ end }}{{- end }}<|im_start|>assistant
"""

PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 8192
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
'''

modelfile_text = f"FROM {gguf_file}\n\n" + MODELFILE_BODY
(GGUF_DIR / "Modelfile.colab").write_text(modelfile_text, encoding="utf-8")
!ollama create smartcampus-vision-8b -f "{GGUF_DIR / 'Modelfile.colab'}"
!ollama run smartcampus-vision-8b "3aslema, chnowa les parcours fi l'IIT?"
proc.terminate()
